In [1]:
import pickle
import os
import numpy as np
import pandas as pd

from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.models.ctm import CombinedTM
from sentence_transformers import SentenceTransformer

In [ ]:
# obligé de reprendre les données ici car bert n'a pas été entrainé sur dataset_final.pkl, donc j'avais un problème de matching entre les embeddings et les textes
df = pd.read_csv("../data/preprocessed/reviews_trust_clean.csv")
df_sans_contexte = pd.read_csv("../data/preprocessed/reviews_trust_clean_stopwords_supprimer.csv")

# on retire quelques stopwords supplémentaires
stopwords_custom = {
    'montre', 'montres', 'boucle', 'boucles', 'oreilles', 'oreille', 'paire', 'paires', 'lunettes', 'bracelet', 'bracelets', 'collier', 'iphone', 'téléphone', 'pendentif', 'robot', 'robots', 'aspirateur', 'baskets', 'basket', 'chaussure', 'chaussures', 'sandales', 'plantes', 'plante', 'arbres', 'arbre', 'bulbes', 'willemse', 'sommiers', 'jardin', 'bague', 'lampe', 'lampes', 'abat-jour', 'abat jour', 'lampadaire', 'parfum', 'shampoing', 'shampooing', 'shampooings', 'cheveux', 'masque', 'masques', 'crème', 'élastiques', 'manteau', 'bougie', 'cadre', 'écouteurs', 'vélo', 'robe', 'vêtements', 'bijoux', 'sac', 'portable', 'clio', 'luminaire', 'oreillette', 'induction', 'écouteur', 'couette', 'samsung', 'téléphones', 'smartcase', 'abat', 'apple', 'watch', 'shirt', 'tee', 'chemise', 'shirts', 'hortensias', 'orchidée', 'sacs', 'plant', 'chaises', 'lacoste', 'polo', 'pantalon', 'jean', 'jeans', 'sneakers', 'lunette', 'écran', 'tablette', 'tablettes', 'table', 'tables', 'chemisier', 'pulls', 'pull', 'trotinettes', 'trotinette', 'chaussons', 'chausson', 'brosse', 'brosses', 'crèmes', 'gel', 'gels', 'parfums', 'robe', 'robes', 'sacoches', 'sacoche', 'vestes', 'veste'
}

def clean_custom(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in stopwords_custom]
    return " ".join(tokens)

df_sans_contexte["clean_comment"] = df_sans_contexte["clean_comment"].apply(clean_custom)
df["clean_comment"] = df["clean_comment"].apply(clean_custom)

texts_sans_contexte = df_sans_contexte["clean_comment"].astype(str).tolist()
texts_avec_contexte = df["clean_comment"]

# Si on modifie encore les stopwords et qu'on veut regénérer les embeddings, décommentez le code ci-dessous
# --------- Pour regénerer les embeddings, décommentez le code ci-dessous ---------
# model = SentenceTransformer('dangvantuan/french-document-embedding', trust_remote_code=True)
# X_ctx = model.encode(
#     texts_avec_contexte,
#     batch_size=32,
#     show_progress_bar=True
# )
# with open("../data/embeddings/emb_sbert_fr_ctm.pkl", "wb") as f:
#     pickle.dump(X_ctx, f)

# --------- Pour regénerer les embeddings, décommentez le code ci-dessus ---------

with open("../data/embeddings/emb_sbert_fr_ctm.pkl", "rb") as f:
    X_ctx = pickle.load(f)
    
X_ctx = np.asarray(X_ctx)
assert X_ctx.shape[0] == len(texts_avec_contexte), "Mismatch nb docs vs nb embeddings"

In [9]:
tp = TopicModelDataPreparation("camembert-base")

training_dataset = tp.fit(
    text_for_contextual=texts_avec_contexte,
    text_for_bow=texts_sans_contexte,
    custom_embeddings=X_ctx
)

bow_size = len(tp.vocab)
ctx_size = X_ctx.shape[1]
bow_size, ctx_size

(20667, 768)

In [10]:
K = 30 # nombre de topics
ctm = CombinedTM(
    bow_size=bow_size,
    contextual_size=ctx_size,
    n_components=K,
    num_epochs=20,
    num_data_loader_workers=0
)

ctm.fit(training_dataset) # entraînement

Epoch: [20/20]	 Seen Samples: [300800/301800]	Train Loss: 229.82593760388963	Time: 0:00:04.179370: : 20it [01:24,  4.21s/it]
100%|██████████| 236/236 [00:02<00:00, 83.56it/s]


In [11]:
topics_words = ctm.get_topic_lists(20)  #  le nombre de mots à afficher par topic
for k, words in enumerate(topics_words):
    print(f"Topic {k}: {', '.join(words)}")

Topic 0: prix, frais, trouve, marque, produits, autres, moins, marques, port, acheter, sites, ventes, cher, ailleurs, parfois, souvent, affichés, 50, pense, disant
Topic 1: falloir, gain, obligation, courage, réceptionnés, suit, commencent, revenu, penser, plaindre, premiere, chargé, trouvent, manifestement, lecture, écrans, buffet, cinq, compensation, sérieusement
Topic 2: tenté, aimé, pensent, obligation, bye, réponde, croix, apercevoir, apparaît, venant, requête, impose, osent, feront, surement, envoyant, revenu, trouvent, oblige, falloir
Topic 3: commandé, déçue, article, déception, bonjour, reçu, déçu, première, commandés, étiquette, articles, renvoyer, comment, erreur, retourner, modèle, moment, cadeau, mauvais, celle
Topic 4: rapide, rapidement, tôt, avance, bonne, emballé, satisfaite, super, rapidité, excellent, prévue, très, emballés, rapport, présentation, prévu, recommande, qualité, produit, date
Topic 5: long, longue, délai, délais, delai, peu, longs, crise, respecté, sanit

In [12]:
doc_topic = ctm.get_doc_topic_distribution(training_dataset)  # shape (n_docs, K)
df["topic_id"] = np.argmax(doc_topic, axis=1)
df["topic_confidence"] = doc_topic.max(axis=1)

100%|██████████| 236/236 [00:02<00:00, 83.52it/s]


In [13]:
os.makedirs("./artifacts/ctm", exist_ok=True)
os.makedirs("./artifacts/ctm/exports", exist_ok=True)

# ctm.save(models_dir="./artifacts/ctm/model")

# with open("./artifacts/ctm/vocab.pkl", "wb") as f:
#     pickle.dump(tp.vocab, f)

pd.DataFrame({
    "topic_id": np.arange(len(topics_words)),
    "top_words": [", ".join(w) for w in topics_words]
}).to_csv("./artifacts/ctm/exports/topics_top_words.csv", index=False)

np.save("./artifacts/ctm/exports/doc_topic.npy", doc_topic)

df.to_csv("./artifacts/ctm/exports/reviews_with_topics.csv", index=False)

In [14]:
topic_counts = df["topic_id"].value_counts().sort_index()
display(topic_counts)

topic_id
0     456
1     262
2     421
3     677
4     615
5     812
6     770
7     624
8     317
9     276
10    647
11    245
12    621
13    690
14    109
15    587
16    292
17    615
18    702
19    950
20    146
21    617
22    696
23    725
24    543
25    342
26    231
27    141
28     53
29    908
Name: count, dtype: int64

In [27]:
def examples_for_topic(t, n=5):
    return df[df["topic_id"]==t]["clean_comment"].head(n).tolist()

for t in range(K):
    ex = examples_for_topic(t, 3)
    print(f"\n=== Topic {t} ===")
    for e in ex:
        print("-", e[:200])


=== Topic 0 ===
- commande passée pour une vente , livraison 15 nous après la date prévue , déjà 6 semaines après l'achat , sur 3 produits , 2 manquants . pas pro , pas sérieux , aucun geste commercial hormis le rembou
- annulation de commande après 2 mois d ’ attente dans un geste et sans explication . retard de livraison et report à 3 reprise pour se résultat incompréhensible et inadmissible je recommande pas le sit
- bonjour , commande fait sur showroom n° 233332467 avec un point relais défini . cette commande a été déviée de son point d'origine de plus de 6km par le bon vouloir du transporteur ? ? réponse de show

=== Topic 1 ===
- j'ai fait la commande d'une sacoche et d'un pull , il y a de ça 1 mois et demi , déjà le temps d'attente est énorme , mais en sois ce n'est pas si grave.par contre sur mon colis à 120 euros , je n'ai 
- j'ai commandé des tapis il y a deux mois , livrables début mai . début mai problèmes techniques , retard dans le traitement chez showroom . puis livrais

In [28]:
#Évaluer la diversité des topics
def topic_diversity(topic_words):
    """
    Calculate topic diversity: proportion of unique words among top words of all topics.
    topic_words : list of lists, each sublist contains the top words of a topic.
    """
    top_words = []
    for words in topic_words:
        top_words.extend(words)

    unique_words = set(top_words)
    diversity = len(unique_words) / len(top_words)
    return diversity
diversity_score = topic_diversity(topics_words)

print(f"Diversité des topics: {diversity_score:.2f}") # closer to 1 : each topic uses unique words(good diversity)

Diversité des topics: 0.77


In [29]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# texts : liste de documents, chaque document = liste de tokens
# topic_words : liste de listes des mots top de chaque topic
texts = df_sans_contexte["clean_comment"].apply(lambda x: x.split()).tolist()
dictionary = Dictionary(texts)

cm = CoherenceModel(
    topics=topics_words,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = cm.get_coherence()

print("Score de cohérence:", coherence_score)

Score de cohérence: 0.5348584450118433


In [31]:
for c in df[df["topic_id"] == 0]["Commentaire"]:
    print(c, "\n---\n")

Commande passée pour une vente Lacoste , livraison 15 nous après la date prévue , déjà 6 semaines après l'achat , sur 3 produits , 2 manquants . Pas pro , pas sérieux , aucun geste commercial hormis le remboursement promis des 2 articles manquants . 
---

Annulation de commande après 2 mois d ’ attente dans un geste et sans explication . Retard de livraison et report à 3 reprise pour se résultat incompréhensible et inadmissible je recommande pas le site mieux vaut aller sur vente privée au mon d on attend mais on a sa commande 
---

Bonjour , Commande fait sur Showroom n° 233332467 avec un point relais défini . Cette commande a été déviée de son point d'origine de plus de 6Km par le bon vouloir du transporteur ? ? Réponse de Showroom n'allez pas le chercher et on vous remboursera ? ? Quelle est l'interré de commander d'attendre plus d'un mois pour le remboursement . Sur Amazon l'ecoute et le respect du client est leur priorité.Merci de prendre exemple 
---

Pour commander il n ' y a pa

In [ ]:
# Regroupement des topics
qualite_produit_topics = {3, 6, 18, 19, 23, 29}
livraison_topics = {5, 10, 24, 26, 27}
service_client_topics = {7, 8, 13, 14, 28}

df['label'] = np.nan

df.loc[df['topic_id'].isin(qualite_produit_topics), 'label'] = 0
df.loc[df['topic_id'].isin(livraison_topics), 'label'] = 1
df.loc[df['topic_id'].isin(service_client_topics), 'label'] = 2

df['label'] = df['label'].astype('Int64')  # int nullable
df['label'].value_counts(dropna=False)

label
<NA>    6191
0       4732
1       2374
2       1793
Name: count, dtype: Int64

Batches:   0%|          | 0/223 [00:00<?, ?it/s]

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


              precision    recall  f1-score   support

         0.0      0.953     0.864     0.906       946
         1.0      0.796     0.819     0.807       475
         2.0      0.756     0.914     0.827       359

    accuracy                          0.862      1780
   macro avg      0.835     0.865     0.847      1780
weighted avg      0.871     0.862     0.864      1780



Classe prédite,0.0,1.0,2.0
Classe réelle,,,
0,817,76,53
1,33,389,53
2,7,24,328


,text,label_true,label_pred,correct
1,j'ai du leurs écrire pendant 2 mois pour me faire rembourser et en plis je n'ai reçu que la moitié . je n'ai jamais reçu ma commande et le service client vous prends pour des… je ne sais même pas s'il s'agit de réels personnes . je ne commanderai plus jamais .,0,2.0,False
3,sur 5 commande jai eu 3 problèmes soit produit manquant soit apres 2 mois d'attente 2 jours avant la livraison ma commande a etait annulé ... ou reception d'un autre article qui n'était pas le mien ... j'hésite a recommander ...,0,1.0,False
6,erreur de livraison . pieces manquantes . colis incomplet .,0,1.0,False
8,j ’ attends depuis des semaines le remboursement de 2 articles retournés ....,0,2.0,False
21,"commande livrée très vite ce qui est très agréable , car bien souvent les délais sont trop longs",0,1.0,False
39,"une catastrophe , le colis est arrivé ouvert et en mauvais état . les produits avaient fuit et étaient a même le carton . une honte .",1,0.0,False
40,"des années que je commande sur veepee et toujours au top . quand je commande chez eux je n'ai jamais été déçu , ils ne se trompent pas dans l'article comme shorommprive trois fois ils se sont trompé de commande . je reçois la plupart toujours à l'avance mais rarement en retard . merci",0,2.0,False
41,une commande passée il y a 4mois . je viens tout juste de recevoir un mail pour me dire que je vais être remboursé et qu ’ ils n ’ allaient pas me livrer ... un service déplorable . je déconseille fortement .,1,2.0,False
44,"commande effectuée le 17 décembre , soit disant expédiée le 20 . j ’ ai contacté le service client par e-mail et j ’ ai eu la chance d ’ obtenir une réponse type « ils effectuent des recherches et me tiennent au courant » . aujourd ’ hui , 2 janvier , aucune nouvelle de mon colis . mon compte en banque a bien sûr été débité depuis longtemps . je extrêmement déçue par l ’ évolution de ce site de vente en ligne .",1,2.0,False
50,"retard sur la livraison mais assez satisfaite sur le produit . seul regret , j ’ aurai du commander taille 54 au lieu de 52",0,1.0,False
